# Notebook zur erstellung des Datasets

In [2]:
import os
import random
import json
import pandas as pd
import numpy as np
from pathlib import Path

### Parameter

In [3]:
DATA_DIR = "../sensor_and_video_data"
LABELED_DIR = "../Labeled"
SEED = 42
# Files expeted in a run folder for it to be considered complete
EXPECTED_FILES = ["selected_peaks", "synch_data", "Timestamps"]

CONVERSION_FACTOR = 2.4  # To convert from video frames to sensor samples


# Split ratios
TRAIN_SPLIT = 0.7
VAL_SPLIT = 0.15
TEST_SPLIT = 0.15


### Helper functions

In [4]:
def check_run_complete(folder, keywords):
    files = os.listdir(os.path.join(DATA_DIR, folder))

    for word in keywords:
        if not any(word in file for file in files):
            return False
    return True

### Runweise Aufteilung in Train/Validation/Test + auschließen von unvollständigen Runs

In [5]:

# Collect all run directories
runs = [
    d for d in os.listdir(DATA_DIR)
    if os.path.isdir(os.path.join(DATA_DIR, d))
]

# Exclude incomplete runs
for run in runs[:]:
    if not check_run_complete(run, EXPECTED_FILES):
        runs.remove(run)
        print(f"Removed incomplete run: {run}")

print(f"Found {len(runs)} runs.")

# Shuffle runs for randomness
random.seed(SEED)
random.shuffle(runs)

# Split into train, val, test
num_runs = len(runs)
train_split = int(TRAIN_SPLIT * num_runs)
val_split = int((TRAIN_SPLIT + VAL_SPLIT) * num_runs)
train_runs = runs[:train_split]
val_runs = runs[train_split:val_split]
test_runs = runs[val_split:]

print(train_runs)
print(val_runs)
print(test_runs)

Found 16 runs.
['0721_0853_noheadsensor', '0724_0723', '0720_0828', '0720_0858', '0802_0800', '0725_0709', '0727_0812', '0721_0928_noheadsensor', '0713_0903', '0713_0932', '0801_0758']
['0803_0734', '0720_0748']
['0727_0735', '0713_0833', '0717_0717']


### Funktion zur Erstellen des Label Vektors

One Hot-Encoding. Bezieht sich auf Video Frames und wird nur Umgerechnet !!!

In [6]:
def create_label_vector(run_file, len_of_run):
    file_path = os.path.join(LABELED_DIR, run_file)
    if os.path.isfile(file_path):
        print(f"Label file exists for {run_file}")
        label_vector = ["nothing"] * len_of_run

        with open(file_path, 'r') as f:
            data = json.load(f)

        raw = data["button_presses"]
        # *2.4 to convert from video frames to sensor samples
        sequence_width_half = int(int(data["sequence_width"])*CONVERSION_FACTOR) // 2

        button_presses = []
        for entry in raw.split(";"):
            entry = entry.strip()
            if not entry:
                continue
            label, idx = entry.split(":")
            button_presses.append((int(float(idx.strip()) * CONVERSION_FACTOR), label.strip())) # convert from video frames to sensor samples

        for idx, label in button_presses:
            if 0 <= idx < len_of_run:
                label_vector[idx-sequence_width_half : idx + sequence_width_half + 1] = [label] * (2 * sequence_width_half + 1)
        one_hot_labels = pd.get_dummies(label_vector)
        return one_hot_labels
        
    else:
        print(f"Label file missing for {run_file}")


create_label_vector("0717_0717_sequences.json", 5000)
    

Label file exists for 0717_0717_sequences.json


,Jumping,nothing
0,False,True
1,False,True
2,False,True
3,False,True
4,False,True
...,...,...
4995,False,True
4996,False,True
4997,False,True
4998,False,True


### Funktion zur Synchronisation von Video- und Sensordaten

Synchronisation des Videos und der Sensordaten durch die Dateie synch_data.json und selected_peaks.json. In selected_peaks.json sind die PacketCounter werte zu finden and denen die verschiedenen Sensoren geklopft wurden. In Timestamps.json sind die dazugehörigen Video Frames. Start des Sliding Windows nach klopfen des letzten Sensors

In [7]:
def synch_data(run_file):
    '''Function which returns the start values for the video file/label vector and the sensor data file.
    when startet from these values both data streams are synchronized. Values are returned in number of samples, NOT 
    video frames.'''

    file_path_synch = os.path.join(DATA_DIR, run_file, "synch_data.json")
    file_path_selected_peaks = os.path.join(DATA_DIR, run_file, "selected_peaks.json")

    # Load synch data
    if os.path.isfile(file_path_synch) and os.path.isfile(file_path_selected_peaks):
        print(f"Synch file and selected peaks file exists for {run_file}")

        with open(file_path_synch, 'r') as f:
            data_synch = json.load(f)

        with open(file_path_selected_peaks, 'r') as f:
            data_selected_peaks = json.load(f)

        # Extract timestamps and convert to sensor samples if needed
        sensor_timestamps_synch_data =  [int(data_synch["Head Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR),
                                        int(data_synch["Wrist Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR),
                                        int(data_synch["Seat Sensor Video (Timestamp) in Frames"]*CONVERSION_FACTOR)]
        sensor_timestamps_selected_peaks = [data_selected_peaks["XSens"]["Head"],
                                            data_selected_peaks["XSens"]["Wrist"],
                                            data_selected_peaks["XSens"]["Seat"]]


        # Determine start values -> the last sensor that was tapped determines the start value for both streams
        start_value_video = max(sensor_timestamps_synch_data)
        start_value_sensor = max(sensor_timestamps_selected_peaks)
        print(f"Start value video/label vector: {start_value_video}, Start value sensor: {start_value_sensor}")

        return start_value_video, start_value_sensor

    else:
        print(f"Synch or selected peaks file missing for {run_file}")
        return

    
synch_data("0717_0717")   


Synch file and selected peaks file exists for 0717_0717
Start value video/label vector: 2544, Start value sensor: 1344


(2544, 1344)

### Funktion um Daten zu laden und in einem Dataframe zusammenzufassen

In [14]:
def load_and_merge_data(run_file,
                        sensor_prefixes=("Wrist", "Head", "Seat"),
                        time_col="PacketCounter",
                        allow_missing=False,
                        fill_missing_with_zeros=False,
                        channels_per_sensor=9):
    
    '''Load and merge sensor data from multiple CSV files in a run folder. If one of the sensor files is missing,
    the user can decide to either raise an error, or fill the missing data with zeros.'''
    
    # Get all csv files in the run folder
    run_path = os.path.join(DATA_DIR, run_file)
    run_path = Path(run_path)
    files = list(run_path.glob("*.csv"))

    data_frames = {}
    base_time = None
    # Columns to be used because some csv files have a , at the end which causes issues
    # also to exclude SampleTimeFine because it is not needed
    cols = ["PacketCounter","Euler_X","Euler_Y","Euler_Z",
        "Acc_X","Acc_Y","Acc_Z","Gyr_X","Gyr_Y","Gyr_Z"]

    for prefix in sensor_prefixes:
        matches = [f for f in files if f.name.startswith(prefix)]
         
        # Handling when files are missing.
        if len(matches) == 0:
            if not allow_missing:
                print(f"No file found for sensor '{prefix}' in run '{run_file}'. Returning empty DataFrame.")
                return pd.DataFrame()  # Return empty DataFrame
            
            if fill_missing_with_zeros:
                print(f"Filling missing sensor data for '{prefix}' with zeros.")
                if base_time is None:
                    raise ValueError("Base time is not set. Cannot create zero-filled DataFrame.")
                dummy = pd.DataFrame(
                    np.zeros((len(base_time), channels_per_sensor)),
                    columns=[f"{prefix.lower()}_ch{i}" for i in range(channels_per_sensor)]
                )
                dummy[time_col] = base_time
                data_frames[prefix.lower()] = dummy
                continue
        
        # read csv file, skip first row because it contains separator info
        df = pd.read_csv(matches[0], skiprows=1, usecols=cols)
        df = df.add_prefix(f"{prefix.lower()}_")
        df = df.rename(columns={f"{prefix.lower()}_{time_col}": time_col})
        data_frames[prefix.lower()] = df

        if base_time is None:
            base_time = df[time_col]

    # Merge all data frames on the time column
    merged = data_frames[sensor_prefixes[0].lower()]
    for prefix in sensor_prefixes[1:]:
        if prefix.lower() in data_frames:
            merged = pd.merge(merged, data_frames[prefix.lower()], on=time_col)

    merged.set_index(time_col, inplace=True)
    return merged

merged = load_and_merge_data("0717_0717")
if not merged.empty:
    display(merged.head())

   

   

,wrist_Euler_X,wrist_Euler_Y,wrist_Euler_Z,wrist_Acc_X,wrist_Acc_Y,wrist_Acc_Z,wrist_Gyr_X,wrist_Gyr_Y,wrist_Gyr_Z,head_Euler_X,...,head_Gyr_Z,seat_Euler_X,seat_Euler_Y,seat_Euler_Z,seat_Acc_X,seat_Acc_Y,seat_Acc_Z,seat_Gyr_X,seat_Gyr_Y,seat_Gyr_Z
PacketCounter,,,,,,,,,,,,,,,,,,,,,
0,-95.135757,-12.468658,-0.610755,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000,-3.287605,...,0.000000,6.620411,-60.786186,-148.595367,0.000000,0.000000,0.000000,0.000000,0.000000,0.000000
1,-95.156738,-12.449160,-0.620541,1.975612,-9.474150,-1.029136,-2.771306,0.933088,2.433237,-3.280030,...,-0.466768,6.624383,-60.786949,-148.588867,8.689031,0.585431,4.818748,1.158954,-0.045952,0.388478
2,-95.346146,-12.320950,-0.578585,1.979963,-9.565908,-1.090644,-2.128517,1.112823,2.419098,-3.318460,...,-0.775010,6.683638,-60.805813,-148.647934,8.678101,0.572914,4.791139,1.060596,-0.034451,0.396649
3,-95.354042,-12.302137,-0.593267,2.009529,-9.571880,-1.123894,-1.321566,1.502108,2.408673,-3.319360,...,-1.222542,6.687258,-60.806927,-148.641495,8.672672,0.582342,4.816631,1.108110,-0.088548,0.390143
4,-95.602249,-12.153739,-0.595800,2.038433,-9.561345,-1.124489,-0.532469,1.806277,2.193334,-3.223926,...,-1.299974,6.740061,-60.821339,-148.680084,8.685561,0.546005,4.825630,1.075350,0.032671,0.265479
